# Compute Cumulative Abnormal Returns (CAR)

This notebook computes cumulative abnormal returns (CAR) for each event.

CAR is defined as the sum of abnormal returns over a specified event window.

We compute:

- CAR[0,1] = AR₀ + AR₁
- CAR[0,3] = AR₀ + AR₁ + AR₂ + AR₃

These measures capture short-term market reactions to earnings announcements.

The input dataset contains abnormal returns aligned in event time (`t`).

In [1]:
import pandas as pd
from pathlib import Path

## 1. Define file paths

In [2]:
event_ar_path = Path("../data/processed/event_abnormal_returns.csv")

output_path = Path("../data/processed/event_CAR.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

## 2. Load dataset

In [3]:
event_ar = pd.read_csv(event_ar_path)

print("event_ar shape:", event_ar.shape)
display(event_ar.head())

event_ar shape: (24628, 16)


,event_id,ticker,file_name,call_datetime_et,after_market_close_et,event_trading_day_final,Date,t,Adj Close,index_adj_close,return,market_return,alpha,beta,expected_return,abnormal_return
0,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-05,-120,25.767363,5139.939941,0.006629,0.006736,-0.000387,1.102993,0.007043,-0.000413
1,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-06,-119,25.823435,5056.439941,0.002176,-0.016245,-0.000387,1.102993,-0.018305,0.020481
2,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-07,-118,25.910910,5043.540039,0.003387,-0.002551,-0.000387,1.102993,-0.003201,0.006588
3,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-10,-117,26.852962,5101.799805,0.036357,0.011551,-0.000387,1.102993,0.012354,0.024003
4,1,AAPL,2016-Jan-26-AAPL.txt,2016-01-26T17:00:00-05:00,True,2016-01-27,2015-08-11,-116,25.455585,5036.790039,-0.052038,-0.012743,-0.000387,1.102993,-0.014442,-0.037597


## 3. Ensure correct sorting

Sorting by event and event-time is critical before aggregation.

In [4]:
event_ar = event_ar.sort_values(["event_id", "t"]).reset_index(drop=True)

## 4. Validate required windows exist

We check that each event has:
- t = 0,1 for CAR[0,1]
- t = 0,1,2,3 for CAR[0,3]

In [5]:
required_t_car01 = {0, 1}
required_t_car03 = {0, 1, 2, 3}

event_t_sets = event_ar.groupby("event_id")["t"].apply(set)

car01_valid = event_t_sets.apply(lambda s: required_t_car01.issubset(s))
car03_valid = event_t_sets.apply(lambda s: required_t_car03.issubset(s))

print("Events with valid CAR[0,1]:", car01_valid.sum())
print("Events with valid CAR[0,3]:", car03_valid.sum())

Events with valid CAR[0,1]: 188
Events with valid CAR[0,3]: 188


## 5. Compute CAR[0,1]

In [6]:
car01 = (
    event_ar[event_ar["t"].isin([0, 1])]
    .groupby("event_id")["abnormal_return"]
    .sum()
    .reset_index(name="CAR_01")
)

## 6. Compute CAR[0,3]

In [7]:
car03 = (
    event_ar[event_ar["t"].isin([0, 1, 2, 3])]
    .groupby("event_id")["abnormal_return"]
    .sum()
    .reset_index(name="CAR_03")
)

## 7. Merge CAR measures

In [8]:
car_df = car01.merge(car03, on="event_id", how="inner")

print("CAR dataset shape:", car_df.shape)
display(car_df.head())

CAR dataset shape: (188, 3)


,event_id,CAR_01,CAR_03
0,1,-0.043240,-0.045063
1,2,-0.074182,-0.089002
2,3,0.071901,0.084408
3,4,-0.023161,-0.029068
4,5,0.055093,0.063649


## 8. Attach event-level metadata

We add ticker and event date for interpretability.

In [9]:
event_info = (
    event_ar[["event_id", "ticker", "event_trading_day_final"]]
    .drop_duplicates()
)

car_df = car_df.merge(event_info, on="event_id", how="left")

display(car_df.head())

,event_id,CAR_01,CAR_03,ticker,event_trading_day_final
0,1,-0.043240,-0.045063,AAPL,2016-01-27
1,2,-0.074182,-0.089002,AAPL,2016-04-27
2,3,0.071901,0.084408,AAPL,2016-07-27
3,4,-0.023161,-0.029068,AAPL,2016-10-26
4,5,0.055093,0.063649,AAPL,2017-02-01


## 9. Validate CAR results

In [10]:
print("Missing CAR_01:", car_df["CAR_01"].isna().sum())
print("Missing CAR_03:", car_df["CAR_03"].isna().sum())

assert car_df["CAR_01"].notna().all(), "Missing CAR_01 values"
assert car_df["CAR_03"].notna().all(), "Missing CAR_03 values"

Missing CAR_01: 0
Missing CAR_03: 0


## 10. Inspect CAR distribution

In [11]:
display(car_df[["CAR_01", "CAR_03"]].describe())

,CAR_01,CAR_03
count,188.000000,188.000000
mean,0.009057,0.007652
std,0.082417,0.093499
min,-0.268187,-0.311183
25%,-0.035060,-0.038219
50%,0.009472,0.004790
75%,0.044508,0.054605
max,0.395444,0.477627


## 11. Check extreme CAR values

Large CARs can occur but should be inspected.

In [12]:
extreme_car = car_df[
    (car_df["CAR_03"].abs() > 0.5) | (car_df["CAR_01"].abs() > 0.5)
]

print("Extreme CAR observations:", len(extreme_car))
display(extreme_car.head(20))

Extreme CAR observations: 0


,event_id,CAR_01,CAR_03,ticker,event_trading_day_final


## 12. Save output

In [13]:
car_df = car_df.sort_values("event_id").reset_index(drop=True)

car_df.to_csv(output_path, index=False)

print("Saved:")
print(output_path)

Saved:
../data/processed/event_CAR.csv


## 13. Validation summary

This confirms:
- number of events
- CAR coverage
- missing values

In [14]:
summary = {
    "n_events": int(car_df["event_id"].nunique()),
    "n_rows": len(car_df),
    "missing_car_01": int(car_df["CAR_01"].isna().sum()),
    "missing_car_03": int(car_df["CAR_03"].isna().sum()),
    "extreme_car_count": int(len(extreme_car)),
}

summary_df = pd.DataFrame(summary.items(), columns=["check", "value"])
display(summary_df)

,check,value
0,n_events,188
1,n_rows,188
2,missing_car_01,0
3,missing_car_03,0
4,extreme_car_count,0


## Conclusion

This notebook computed cumulative abnormal returns for each event:

- CAR[0,1]
- CAR[0,3]

These variables represent the short-term market reaction to earnings calls and will serve as key dependent variables in the final regression analysis.

The dataset is now ready for:
- volatility analysis
- sentiment merging
- hypothesis testing